# Arbitrary States — CUDA-Q Implementation

**Primitive:** Arbitrary single-qubit state preparation via RY-RZ rotations

**SETUP (must match the Qiskit notebook exactly for results to line up)**
- N QUBITS: 2 (prepared independently, no entanglement)
- QUBIT 0: RZ(phi0) . RY(theta0), theta0 = pi/3, phi0 = pi/4
- QUBIT 1: RZ(phi1) . RY(theta1), theta1 = 2*pi/3, phi1 = pi/2
- QUBIT ORDERING (contract): qubit 0 = rightmost bit in printed bitstring
- SHOTS: 10000, SEED: 42
- GLOBAL PHASE: normalized so first nonzero amplitude (ascending bitstring order) is real and positive

**Theory recap:** identical to the Qiskit version -- |psi> = RZ(phi) RY(theta) |0>, with RY
setting the |0>/|1> amplitude split and RZ setting the relative phase.

**Important ordering warning:** CUDA-Q's native state-vector indexing treats qubit 0 as the
**most-significant** bit (opposite of Qiskit's rightmost/least-significant convention). This
notebook explicitly reverses bit order after extracting the state so the final printed
dictionary matches the SETUP convention and is directly diffable against `QISKIT_RESULT`.
Unlike Basic States or Superposition, this primitive has genuinely different probabilities
per basis state, so the reordering step here has a real, visible effect -- watch this one
closely when you diff the two dicts.

**IMPORTANT: Run every cell in order, top to bottom** (Kernel -> Restart & Run All).

In [ ]:
# Run this cell first if you get ModuleNotFoundError.
# Correct package name is "cudaq" (NOT "cuda-quantum" -- that name has no prebuilt wheel and breaks pip).
# Requires pip >= 24.0, and Linux (x86_64/aarch64) or macOS (arm64/Apple Silicon). Windows needs WSL.
!pip install --upgrade pip -q
!pip install -q cudaq
# Restart the kernel after this finishes (Kernel -> Restart Kernel), then Run All.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cudaq

N_QUBITS = 2
SHOTS = 10000
SEED = 42

cudaq.set_random_seed(SEED)

# Rotation angles: (theta, phi) per qubit, applied as RZ(phi) . RY(theta) -- same values as Qiskit
THETA0, PHI0 = np.pi / 3, np.pi / 4          # qubit 0
THETA1, PHI1 = 2 * np.pi / 3, np.pi / 2      # qubit 1
print(f"Qubit 0: theta={THETA0:.4f}, phi={PHI0:.4f}")
print(f"Qubit 1: theta={THETA1:.4f}, phi={PHI1:.4f}")

## 1. Build the kernel explicitly, gate by gate

In [ ]:
@cudaq.kernel
def arbitrary_states_kernel(theta0: float, phi0: float, theta1: float, phi1: float):
    q = cudaq.qvector(2)
    # Arbitrary-state prep on qubit 0: RY sets amplitude split, RZ sets relative phase
    ry(theta0, q[0])
    rz(phi0, q[0])
    # Arbitrary-state prep on qubit 1: same decomposition, independent angles
    ry(theta1, q[1])
    rz(phi1, q[1])

**Note:** angles are passed as kernel arguments here (CUDA-Q kernels support typed float
arguments), rather than hardcoded, since RY/RZ take continuous parameters -- this differs
from Basic States/Superposition where the gate choice itself was hardcoded per qubit count.

## 2. Draw/visualize the circuit

In [ ]:
# Native CUDA-Q text drawing
print(cudaq.draw(arbitrary_states_kernel, THETA0, PHI0, THETA1, PHI1))

In [ ]:
# Matplotlib circuit diagram, styled to resemble the Qiskit mpl output
fig, ax = plt.subplots(figsize=(5, 2))

wire_labels = [f"q_{i}" for i in range(N_QUBITS)]
for i, label in enumerate(wire_labels):
    y = N_QUBITS - 1 - i
    ax.hlines(y, 0, 4, color="black", linewidth=1)
    ax.text(-0.3, y, label, va="center", ha="right", fontsize=10)

gate_labels = {0: ("RY(t0)", "RZ(p0)"), 1: ("RY(t1)", "RZ(p1)")}
for q in range(N_QUBITS):
    y = N_QUBITS - 1 - q
    ry_label, rz_label = gate_labels[q]
    ax.add_patch(plt.Rectangle((0.7, y - 0.25), 0.9, 0.5, facecolor="#f6b26b", edgecolor="black"))
    ax.text(1.15, y, ry_label, va="center", ha="center", fontsize=8, fontweight="bold")
    ax.add_patch(plt.Rectangle((2.0, y - 0.25), 0.9, 0.5, facecolor="#f6b26b", edgecolor="black"))
    ax.text(2.45, y, rz_label, va="center", ha="center", fontsize=8, fontweight="bold")

ax.set_xlim(-0.6, 4)
ax.set_ylim(-0.5, N_QUBITS - 0.5)
ax.axis("off")
ax.set_title("Arbitrary States — CUDA-Q circuit")

fig.savefig("arbitrary_states_cudaq_circuit.png", bbox_inches="tight")
fig

## 3. Explicit matrix representation of the key operator(s)

In [ ]:
def ry_matrix(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s],
                      [s,  c]], dtype=complex)

def rz_matrix(phi):
    return np.array([[np.exp(-1j * phi / 2), 0],
                      [0, np.exp(1j * phi / 2)]], dtype=complex)

for q, (theta, phi) in enumerate([(THETA0, PHI0), (THETA1, PHI1)]):
    print(f"--- Qubit {q}: theta={theta:.4f}, phi={phi:.4f} ---")
    print("RY(theta) matrix:")
    print(np.round(ry_matrix(theta), 4))
    print("RZ(phi) matrix:")
    print(np.round(rz_matrix(phi), 4))
    combined = rz_matrix(phi) @ ry_matrix(theta)
    print("Combined single-qubit operator RZ(phi).RY(theta):")
    print(np.round(combined, 4))
    print()

In [ ]:
# Full 4x4 operator, constructed manually from gate definitions
# (same tensor-product convention as Qiskit: qubit 0 rightmost in basis ordering
# {|00>, |01>, |10>, |11>}).
op_qubit0 = rz_matrix(PHI0) @ ry_matrix(THETA0)
op_qubit1 = rz_matrix(PHI1) @ ry_matrix(THETA1)
full_op = np.kron(op_qubit1, op_qubit0)

print(f"Full {2**N_QUBITS}x{2**N_QUBITS} operator (basis {{|00>, |01>, |10>, |11>}}):")
print(np.round(full_op, 4))

## 4. Exact statevector, reordered + global-phase normalized

In [ ]:
state = cudaq.get_state(arbitrary_states_kernel, THETA0, PHI0, THETA1, PHI1)
raw_amps = np.array(state)

print("Raw CUDA-Q amplitudes (native ordering, qubit0 = MSB):")
print(np.round(raw_amps, 4))

In [ ]:
# --- Explicit reindexing: CUDA-Q native (qubit0=MSB) -> SETUP convention (qubit0=rightmost/LSB) ---
def reorder_cudaq_to_qiskit_convention(amps, n_qubits):
    dim = 2 ** n_qubits
    reordered = np.zeros(dim, dtype=complex)
    for cudaq_index in range(dim):
        cudaq_bits = format(cudaq_index, f"0{n_qubits}b")   # qubit0 = MSB (CUDA-Q native)
        target_bits = cudaq_bits[::-1]                        # qubit0 = rightmost (SETUP)
        target_index = int(target_bits, 2)
        reordered[target_index] = amps[cudaq_index]
    return reordered

amps_reordered = reorder_cudaq_to_qiskit_convention(raw_amps, N_QUBITS)
print("Reordered amplitudes (qubit0 = rightmost, matches SETUP convention):")
print(np.round(amps_reordered, 4))

In [ ]:
# Global phase normalization -- identical method to the Qiskit notebook.
def normalize_global_phase(statevector_data, n_qubits):
    dim = 2 ** n_qubits
    bitstrings = [format(i, f"0{n_qubits}b") for i in range(dim)]
    order = np.argsort(bitstrings)
    for idx in order:
        amp = statevector_data[idx]
        if abs(amp) > 1e-9:
            phase = np.angle(amp)
            corrected = statevector_data * np.exp(-1j * phase)
            return corrected, bitstrings
    return statevector_data, bitstrings

amps_norm, bitstrings = normalize_global_phase(amps_reordered, N_QUBITS)

exact_probs = {}
for i, bs in enumerate(bitstrings):
    exact_probs[bs] = round(abs(amps_norm[i]) ** 2, 4)

exact_probs = dict(sorted(exact_probs.items()))
print("Exact statevector amplitudes (phase-normalized):")
print(np.round(amps_norm, 4))
print()
print("Exact statevector probabilities (qubit0 = rightmost):")
print(exact_probs)

## 5. Measurement: sample, plot histogram

In [ ]:
result = cudaq.sample(arbitrary_states_kernel, THETA0, PHI0, THETA1, PHI1, shots_count=SHOTS)
raw_counts = {bs: result.count(bs) for bs in result}
print("Raw CUDA-Q counts (native bitstring ordering, qubit0 = MSB):")
print(raw_counts)

In [ ]:
# Reverse each sampled bitstring the same way as the statevector, to match SETUP convention
sampled_probs = {}
for bs, c in raw_counts.items():
    reordered_bs = bs[::-1]
    sampled_probs[reordered_bs] = sampled_probs.get(reordered_bs, 0) + c

sampled_probs = {bs: round(c / SHOTS, 4) for bs, c in sampled_probs.items()}
sampled_probs = dict(sorted(sampled_probs.items()))
print(f"Sampled probabilities ({SHOTS} shots, seed={SEED}), qubit0 = rightmost:")
print(sampled_probs)

In [ ]:
# Matplotlib bar chart styled like Qiskit's plot_histogram()
fig2, ax2 = plt.subplots(figsize=(6, 4))
labels = list(sampled_probs.keys())
values = list(sampled_probs.values())

ax2.bar(labels, values, color="#f6b26b", edgecolor="black")
ax2.set_xlabel("Measured bitstring")
ax2.set_ylabel("Probability")
ax2.set_title("Arbitrary States — CUDA-Q measurement histogram")
ax2.set_ylim(0, 1.05)
for i, v in enumerate(values):
    ax2.text(i, v + 0.02, str(v), ha="center", fontsize=9)

fig2.savefig("arbitrary_states_cudaq_histogram.png", bbox_inches="tight")
fig2

## 6. Gate / operation count

In [ ]:
gate_count = 2 * N_QUBITS  # one RY + one RZ per qubit
print(f"Gate/operation count: {gate_count}")

## 7. Convention notes

In [ ]:
print("Qubit ordering: CUDA-Q's cudaq.get_state() and cudaq.sample() natively index/report")
print("bitstrings with qubit 0 as the MOST-significant (leftmost) bit -- the opposite of Qiskit's")
print("convention. This notebook explicitly reversed every bitstring (both for the statevector")
print("amplitudes and the sampled counts) via string reversal / index remapping, so the final")
print("printed results use qubit 0 = rightmost, matching the SETUP convention and the Qiskit output.")
print()
print("Global phase: computed and removed using the identical method as the Qiskit notebook --")
print("first nonzero amplitude in ascending-bitstring order is rotated to be real and positive.")
print("This primitive produces genuinely complex amplitudes (due to the RZ phase rotations), so")
print("this normalization step is NOT a no-op here.")

## Final result (for diffing against QISKIT_RESULT)

In [ ]:
CUDAQ_RESULT = exact_probs
print("CUDAQ_RESULT =", CUDAQ_RESULT)

## Quick diff check
Run this after both notebooks have produced `QISKIT_RESULT` and `CUDAQ_RESULT`.
Expected (hand-computed from the RY angles): `{'00': 0.1875, '01': 0.0625, '10': 0.5625, '11': 0.1875}`

In [ ]:
# QISKIT_RESULT = {'00': 0.1875, '01': 0.0625, '10': 0.5625, '11': 0.1875}  # paste from the Qiskit notebook
#
# all_keys = sorted(set(QISKIT_RESULT) | set(CUDAQ_RESULT))
# max_delta = max(abs(QISKIT_RESULT.get(k, 0) - CUDAQ_RESULT.get(k, 0)) for k in all_keys)
# print("Max probability delta:", max_delta)
# print("Match within tolerance:", max_delta < 1e-3)